# EAT model (based on Wang's paper)

## Data loading, and preparing for EAT

In [15]:
import os
import glob
import torch
import pandas as pd
import soundfile as sf
import torchaudio
from torch.utils.data import Dataset, DataLoader

class DCASEWangDataset(Dataset):
    def __init__(self, root_dir, machine_type, target_length=1024, is_train=True):
        self.root_dir = root_dir
        self.machine_type = machine_type
        self.target_length = target_length
        self.is_train = is_train
        
        # 1. Load the attributes CSV
        csv_path = os.path.join(root_dir, machine_type, "attributes_00.csv")
        self.df = pd.read_csv(csv_path)
        
        # Logic for Domain vs Attribute
        val_cols = [c for c in self.df.columns if c.endswith('v')]
        if len(val_cols) > 0:
            self.df['label_str'] = self.df[val_cols].astype(str).agg('_'.join, axis=1)
            self.task_type = "attribute"
        else:
            if 'domain' in self.df.columns:
                self.df['label_str'] = self.df['domain'].astype(str)
            else:
                self.df['label_str'] = self.df['file_name'].apply(
                    lambda x: "source" if "source" in x else "target"
                )
            self.task_type = "domain"

        unique_labels = sorted(self.df['label_str'].unique())
        self.label_to_id = {label: i for i, label in enumerate(unique_labels)}
        self.num_classes = len(self.label_to_id)
        self.label_lookup = {
            os.path.basename(row['file_name']): self.label_to_id[row['label_str']] 
            for _, row in self.df.iterrows()
        }
        
        subset = "train" if is_train else "test"
        search_path = os.path.join(root_dir, machine_type, subset, "*.wav")
        self.file_list = glob.glob(search_path)
        
    def __len__(self):
        return len(self.file_list)

    # def __getitem__(self, idx):
    #     wav_path = self.file_list[idx]
    #     fname = os.path.basename(wav_path)
        
    #     # Get label from CSV lookup; default to 0 for test files without attributes
    #     label_id = self.label_lookup.get(fname, 0)
        
    #     # Load audio
    #     wav, sr = sf.read(wav_path)
    #     wav = torch.tensor(wav).float()
        
    #     # Resample to 16k (EAT requirement)
    #     if sr != 16000:
    #         wav = torchaudio.functional.resample(wav, sr, 16000)
        
    #     # Normalization
    #     wav = wav - wav.mean()
        
    #     return wav, torch.tensor(label_id)
    
    def __getitem__(self, idx):
        wav_path = self.file_list[idx]
        fname = os.path.basename(wav_path)
        domain_id = 0 if "source" in fname else 1
        label_id = self.label_lookup.get(fname, 0)
        
        wav, sr = sf.read(wav_path)
        wav = torch.tensor(wav).float()
        
        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)
        
        wav = wav - wav.mean()
        
        # Preprocess here so DataLoader returns the final model input
        mel = self.preprocess_audio(wav)
        
        return mel, torch.tensor(label_id), torch.tensor(domain_id)

    def preprocess_audio(wav, sr=16000, target_length=1024):
        # Convert to mel-spectrogram
        mel = torchaudio.compliance.kaldi.fbank(
            wav.unsqueeze(0), htk_compat=True, sample_frequency=sr, use_energy=False,
            window_type='hanning', num_mel_bins=128, dither=0.0, frame_shift=10
        ).unsqueeze(0) # Output shape: [1, T, 128]
        
        # Padding/Truncating to target_length (e.g., 1024)
        n_frames = mel.shape[1]
        if n_frames < target_length:
            mel = torch.nn.ZeroPad2d((0, 0, 0, target_length - n_frames))(mel)
        else:
            mel = mel[:, :target_length, :]
        
        # EAT Global Normalization constants
        return (mel - (-4.268)) / (4.569 * 2)

In [ ]:
# import os
# import glob
# import torch
# import pandas as pd
# import soundfile as sf
# import torchaudio
# from torch.utils.data import Dataset, DataLoader

# class DCASEWangDataset(Dataset):
#     def __init__(self, root_dir, machine_type, target_length=1024, is_train=True):
#         self.root_dir = root_dir
#         self.machine_type = machine_type
#         self.target_length = target_length
#         self.is_train = is_train
        
#         # 1. Load the attributes CSV
#         csv_path = os.path.join(root_dir, machine_type, "attributes_00.csv")
#         self.df = pd.read_csv(csv_path)
        
#         # --- NEW LOGIC FOR DOMAIN VS ATTRIBUTE ---
#         # Check if the CSV has actual attribute columns (ending in 'v')
#         val_cols = [c for c in self.df.columns if c.endswith('v')]
        
#         if len(val_cols) > 0:
#             # OPTION A: Attribute Classification (Wang's primary method)
#             self.df['label_str'] = self.df[val_cols].astype(str).agg('_'.join, axis=1)
#             self.task_type = "attribute"
#         else:
#             # OPTION B: Domain Classification (Wang's fallback for missing info)
#             # Use the 'd' column if it exists, otherwise extract from 'file_name'
#             if 'domain' in self.df.columns:
#                 self.df['label_str'] = self.df['domain'].astype(str)
#             else:
#                 self.df['label_str'] = self.df['file_name'].apply(
#                     lambda x: "source" if "source" in x else "target"
#                 )
#             self.task_type = "domain"

#         # 3. Create Label Map for ArcFace
#         unique_labels = sorted(self.df['label_str'].unique())
#         self.label_to_id = {label: i for i, label in enumerate(unique_labels)}
#         self.num_classes = len(self.label_to_id)
        
#         # 4. Create lookup dictionary
#         self.label_lookup = {
#             os.path.basename(row['file_name']): self.label_to_id[row['label_str']] 
#             for _, row in self.df.iterrows()
#         }
        
#         # 5. Get file list
#         subset = "train" if is_train else "test"
#         search_path = os.path.join(root_dir, machine_type, subset, "*.wav")
#         self.file_list = glob.glob(search_path)

#     def __len__(self):
#         return len(self.file_list)

#     def __getitem__(self, idx):
#         wav_path = self.file_list[idx]
#         fname = os.path.basename(wav_path)
        
#         # Identify domain for scoring/metrics later
#         # (0 for source, 1 for target)
#         domain_id = 0 if "source" in fname else 1
        
#         # Get classification label (Attribute ID or Domain ID depending on task_type)
#         label_id = self.label_lookup.get(fname, 0)
        
#         wav, sr = sf.read(wav_path)
#         wav = torch.tensor(wav).float()
        
#         if sr != 16000:
#             wav = torchaudio.functional.resample(wav, sr, 16000)
        
#         wav = wav - wav.mean()
        
#         # Returning domain_id as well helps for calculating AUC(source) and AUC(target) separately
#         return wav, torch.tensor(label_id), torch.tensor(domain_id)
    
#     def preprocess_audio(wav, sr=16000, target_length=1024):
#         # Convert to mel-spectrogram
#         mel = torchaudio.compliance.kaldi.fbank(
#             wav.unsqueeze(0), htk_compat=True, sample_frequency=sr, use_energy=False,
#             window_type='hanning', num_mel_bins=128, dither=0.0, frame_shift=10
#         ).unsqueeze(0) # Output shape: [1, T, 128]
        
#         # Padding/Truncating to target_length (e.g., 1024)
#         n_frames = mel.shape[1]
#         if n_frames < target_length:
#             mel = torch.nn.ZeroPad2d((0, 0, 0, target_length - n_frames))(mel)
#         else:
#             mel = mel[:, :target_length, :]
        
#         # EAT Global Normalization constants
#         return (mel - (-4.268)) / (4.569 * 2)

In [17]:
# Initialize for Bearing
dataset = DCASEWangDataset("data/dcase2023t2/dev_data/raw", "bearing", is_train=True)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f"Machine: {dataset.machine_type}")
print(f"Total classes to learn: {dataset.num_classes}")
print(f"Class mapping: {dataset.label_to_id}")
print(f"Task type: {dataset.task_type}")
print(f"source files: {sum(1 for f in dataset.file_list if 'source' in f)}")
print(f"target files: {sum(1 for f in dataset.file_list if 'target' in f)}")

Machine: bearing
Total classes to learn: 29
Class mapping: {'11_A': 0, '13_A': 1, '15_A': 2, '16_A': 3, '16_B': 4, '16_C': 5, '16_D': 6, '16_E': 7, '16_F': 8, '16_G': 9, '16_H': 10, '17_A': 11, '19_A': 12, '1_A': 13, '21_A': 14, '23_A': 15, '25_A': 16, '3_A': 17, '5_A': 18, '7_A': 19, '8_A': 20, '8_B': 21, '8_C': 22, '8_D': 23, '8_E': 24, '8_F': 25, '8_G': 26, '8_H': 27, '9_A': 28}
Task type: attribute
source files: 990
target files: 10


In [ ]:
# # 1. Initialize for Bearing (or any machine type)
# # Make sure the path points to your actual data folder
# dataset = DCASEWangDataset("data/dcase2023t2/dev_data/raw", "bearing", is_train=True)
# dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# # 2. Check the metadata extraction
# print(f"--- Dataset Info ---")
# print(f"Machine Type: {dataset.machine_type}")
# print(f"Task Detected: {dataset.task_type}")  # Should say 'attribute' for bearing
# print(f"Total classes (N): {dataset.num_classes}")
# print(f"Mapping (First 3): {list(dataset.label_to_id.items())[:3]}")

# # 3. Test a single batch
# wav, label_id, domain_id = next(iter(dataloader))

# print(f"\n--- Batch Check ---")
# print(f"Waveform shape: {wav.shape}")        # Expected: [batch, samples]
# print(f"Labels (Targets): {label_id}")       # The IDs for ArcFace
# print(f"Domains (0=Src, 1=Trg): {domain_id}") # The domain info for AUC calculation

# # 4. Verify specific domain counts in the whole dataset
# source_count = sum(1 for f in dataset.file_list if "source" in f)
# target_count = sum(1 for f in dataset.file_list if "target" in f)
# print(f"\n--- File Distribution ---")
# print(f"Source files: {source_count}")
# print(f"Target files: {target_count}")

--- Dataset Info ---
Machine Type: bearing
Task Detected: attribute
Total classes (N): 29
Mapping (First 3): [('11_A', 0), ('13_A', 1), ('15_A', 2)]

--- Batch Check ---
Waveform shape: torch.Size([4, 160000])
Labels (Targets): tensor([18,  6,  3, 28])
Domains (0=Src, 1=Trg): tensor([0, 0, 0, 0])

--- File Distribution ---
Source files: 990
Target files: 10


In [ ]:
# import torch.nn as nn
# import torch.nn.functional as F
# import math
# from transformers import AutoModel

# class ArcFaceHead(nn.Module):
#     def __init__(self, in_features, out_features, s=30.0, m=0.50):
#         super(ArcFaceHead, self).__init__()
#         self.in_features = in_features
#         self.out_features = out_features
#         self.s = s # Scaling factor
#         self.m = m # Margin
#         self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
#         nn.init.xavier_uniform_(self.weight)

#     def forward(self, input, label):
#         # 1. Norm the weights and input
#         cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        
#         # 2. Add margin
#         sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
#         phi = cosine * math.cos(self.m) - sine * math.sin(self.m)
        
#         # 3. Only apply margin to the 'correct' class
#         one_hot = torch.zeros(cosine.size(), device=input.device)
#         one_hot.scatter_(1, label.view(-1, 1).long(), 1)
#         output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        
#         return output * self.s

# class WangEATModel(nn.Module):
#     def __init__(self, num_classes):
#         super(WangEATModel, self).__init__()
#         # Load pre-trained EAT (Efficient Audio Transformer)
#         # Assuming you have the 'eat_model' package or local file
#         self.backbone = torch.hub.load('efficient-audio-transformer', 'eat_base', pretrained=True)
        
#         # The EAT-base embedding size is typically 768
#         self.embedding_size = 768 
        
#         # ArcFace Head for attribute/domain classification
#         self.arcface = ArcFaceHead(self.embedding_size, num_classes)

#     def forward(self, x, labels=None):
#         # 1. Extract features (Patch Embeddings)
#         # x input shape: [batch, 1, time, freq]
#         features = self.backbone.extract_features(x) 
        
#         # 2. Global Average Pooling (as per paper)
#         # features shape: [batch, patches, 768] -> [batch, 768]
#         embeddings = features.mean(dim=1)
        
#         if labels is not None:
#             # During training: Return ArcFace logits
#             return self.arcface(embeddings, labels)
        
#         # During inference: Return normalized embeddings for KNN
#         return F.normalize(embeddings)

In [18]:
import torch.nn as nn
import torch.nn.functional as F

class WangEATModel(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone # EAT-base
        self.num_classes = num_classes
        # ArcFace Weight Matrix: [num_classes, embedding_size]
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, 768))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, labels=None, s=30.0, m=0.5):
        # 1. Extract features (returns patch embeddings)
        feats = self.backbone.extract_features(x) 
        
        # 2. Average Pooling (Specific to Wang's Section 2.1)
        # feats shape: [batch, patches, 768] -> [batch, 768]
        pooled_feat = feats.mean(dim=1) 
        
        if labels is not None:
            # 3. ArcFace Logic (Section 2.2)
            cosine = F.linear(F.normalize(pooled_feat), F.normalize(self.weight))
            sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
            phi = cosine * torch.cos(torch.tensor(m)) - sine * torch.sin(torch.tensor(m))
            
            one_hot = torch.zeros(cosine.size(), device=x.device)
            one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
            output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
            return output * s # This goes to CrossEntropyLoss
            
        return pooled_feat # Returns embedding for KNN during inference

In [19]:
from transformers import AutoModel

model_id = "worstchan/EAT-base_epoch30_finetune_AS2M"
device = "cuda" if torch.cuda.is_available() else "cpu"

backbone = AutoModel.from_pretrained(model_id, trust_remote_code=True)

model = WangEATModel(backbone, num_classes=dataset.num_classes).to(device)

model.train()

WangEATModel(
  (backbone): EATModel(
    (model): EAT(
      (local_encoder): PatchEmbed_new(
        (proj): Conv2d(1, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (pos_drop): Dropout(p=0.0, inplace=True)
      (fixed_positional_encoder): FixedPositionalEncoder()
      (blocks): ModuleList(
        (0-11): 12 x AltBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): AltAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (drop_path): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (act): GELU(approximate='none')
            (drop1): Dropout(p=0.0, inplace

In [6]:
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

# --- SETTINGS ---
epochs = 1 # Wang suggests more, but 10 is a good start
lr = 5e-5
device = "cuda" if torch.cuda.is_available() else "cpu"

optimizer = AdamW(model.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss()
scaler = GradScaler()

model.train()
print(f"Starting training for {dataset.machine_type}...")

for epoch in range(epochs):
    total_loss = 0
    for batch_idx, (wavs, labels) in enumerate(dataloader):
        labels = labels.to(device)
        
        # Preprocess on the fly (or you can move this into the Dataset __getitem__)
        mels = torch.stack([preprocess_audio(w) for w in wavs]).to(device)
        
        optimizer.zero_grad()
        
        with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
            # s=30 and m=0.5 are standard Wang/ArcFace hyperparameters
            logits = model(mels, labels=labels, s=30.0, m=0.5)
            loss = criterion(logits, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}")

    print(f"Epoch {epoch+1} Average Loss: {total_loss / len(dataloader):.4f}")

print("Training Complete!")

Starting training for bearing...
Epoch 1 | Batch 0/250 | Loss: 17.6883
Epoch 1 | Batch 50/250 | Loss: 10.3358
Epoch 1 | Batch 100/250 | Loss: 6.2454
Epoch 1 | Batch 150/250 | Loss: 2.0566
Epoch 1 | Batch 200/250 | Loss: 4.6524
Epoch 1 Average Loss: 5.9444
Training Complete!


In [7]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

model.eval()
train_features = []

print("Building Normal Feature Library...")
with torch.no_grad():
    for wavs, _ in dataloader:
        mels = torch.stack([preprocess_audio(w) for w in wavs]).to(device)
        # Without labels, the model returns the 768-dim embedding
        feats = model(mels) 
        train_features.append(feats.cpu().numpy())

train_features = np.vstack(train_features)

# Wang uses k=2 for the k-Nearest Neighbors distance
knn = NearestNeighbors(n_neighbors=1, metric='cosine')
knn.fit(train_features)
print("Library Ready.")

Building Normal Feature Library...
Library Ready.


In [8]:
# 1. Load Test Data
test_dataset = DCASEWangDataset("data/dcase2023t2/dev_data/raw", "bearing", is_train=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

test_scores = []
filenames = []

print("Scoring test files...")
with torch.no_grad():
    for wavs, _ in test_loader:
        mels = torch.stack([preprocess_audio(w) for w in wavs]).to(device)
        feats = model(mels).cpu().numpy()
        
        # Calculate distance to nearest normal neighbor
        distances, _ = knn.kneighbors(feats)
        # Anomaly Score is the mean distance to the k-neighbors
        scores = np.mean(distances, axis=1)
        test_scores.extend(scores)

# Save results to CSV
results = pd.DataFrame({
    'file_name': [os.path.basename(f) for f in test_dataset.file_list],
    'anomaly_score': test_scores
})
results.to_csv(f"results_{dataset.machine_type}_1.csv", index=False)
print("Results saved to CSV!")

Scoring test files...
Results saved to CSV!


In [9]:
import pandas as pd
import numpy as np
from sklearn import metrics

# 1. Load your results
df = pd.read_csv('results_bearing_1.csv')

# 2. Extract Ground Truth from filenames
# In DCASE, 'anomaly' is the positive class (1) and 'normal' is (0)
df['label'] = df['file_name'].apply(lambda x: 1 if 'anomaly' in x else 0)

# 3. Calculate AUC
auc = metrics.roc_auc_score(df['label'], df['anomaly_score'])

# 4. Calculate pAUC (Standard DCASE setting: max_fpr=0.1)
p_auc = metrics.roc_auc_score(df['label'], df['anomaly_score'], max_fpr=0.1)

print(f"Machine Type: Bearing")
print(f"AUC: {auc:.4f}")
print(f"pAUC (fpr=0.1): {p_auc:.4f}")

Machine Type: Bearing
AUC: 0.6851
pAUC (fpr=0.1): 0.5532


In [ ]:
# from transformers import AutoModel

# model_id = "worstchan/EAT-base_epoch30_finetune_AS2M"
# backbone = AutoModel.from_pretrained(model_id, trust_remote_code=True)

# # 1. Setup device
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # 2. Initialize the model 
# # We use the num_classes we got from our dataset earlier (29 for bearing)
# model = WangEATModel(backbone, num_classes=dataset.num_classes).to(device)

# # 3. Get a real batch from your dataloader
# wav, label_id, domain_id = next(iter(dataloader))
# wav = wav.to(device)
# label_id = label_id.to(device)

# # 4. Preprocess (EAT expects Mel Spectrograms, not raw waves)
# # We need to apply the preprocess_audio function to the batch
# # For simplicity in this test, let's process one sample:
# input_mel = preprocess_audio(wav[0]).to(device) # Shape: [1, 1, 1024, 128]

# # 5. Forward Pass
# model.train() # Set to train mode to use ArcFace logic
# logits = model(input_mel, label_id[0].unsqueeze(0))

# print(f"Input shape: {input_mel.shape}")
# print(f"Output (Logits) shape: {logits.shape}") # Should be [1, 29]
# print("Forward pass successful!")

ValueError: not enough values to unpack (expected 3, got 2)

# Finetuning

In [32]:
import torch.nn as nn
import torch.nn.functional as F

class WangEATModel(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone # EAT-base
        self.num_classes = num_classes
        # ArcFace Weight Matrix: [num_classes, embedding_size]
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, 768))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, labels=None, s=30.0, m=0.5):
        # 1. Extract features (returns patch embeddings)
        feats = self.backbone.extract_features(x) 
        
        # 2. Average Pooling (Specific to Wang's Section 2.1)
        # feats shape: [batch, patches, 768] -> [batch, 768]
        pooled_feat = feats.mean(dim=1) 
        
        if labels is not None:
            # 3. ArcFace Logic (Section 2.2)
            cosine = F.linear(F.normalize(pooled_feat), F.normalize(self.weight))
            sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
            phi = cosine * torch.cos(torch.tensor(m)) - sine * torch.sin(torch.tensor(m))
            
            one_hot = torch.zeros(cosine.size(), device=x.device)
            one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
            output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
            return output * s # This goes to CrossEntropyLoss
            
        return pooled_feat # Returns embedding for KNN during inference

In [ ]:
from transformers import AutoModel

model_id = "worstchan/EAT-base_epoch30_finetune_AS2M"
device = "cuda" if torch.cuda.is_available() else "cpu"

backbone = AutoModel.from_pretrained(model_id, trust_remote_code=True)

model = WangEATModel(backbone, num_classes=dataset.num_classes).to(device)

model.train()

WangEATModel(
  (backbone): EATModel(
    (model): EAT(
      (local_encoder): PatchEmbed_new(
        (proj): Conv2d(1, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (pos_drop): Dropout(p=0.0, inplace=True)
      (fixed_positional_encoder): FixedPositionalEncoder()
      (blocks): ModuleList(
        (0-11): 12 x AltBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): AltAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (drop_path): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (act): GELU(approximate='none')
            (drop1): Dropout(p=0.0, inplace

In [34]:
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

# --- SETTINGS ---
epochs = 5 # Wang suggests more, but 10 is a good start
lr = 5e-5
device = "cuda" if torch.cuda.is_available() else "cpu"

optimizer = AdamW(model.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss()
scaler = GradScaler()

model.train()
print(f"Starting training for {dataset.machine_type}...")

for epoch in range(epochs):
    total_loss = 0
    for batch_idx, (wavs, labels) in enumerate(dataloader):
        labels = labels.to(device)
        
        # Preprocess on the fly (or you can move this into the Dataset __getitem__)
        mels = torch.stack([preprocess_audio(w) for w in wavs]).to(device)
        
        optimizer.zero_grad()
        
        with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
            # s=30 and m=0.5 are standard Wang/ArcFace hyperparameters
            logits = model(mels, labels=labels, s=30.0, m=0.5)
            loss = criterion(logits, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}")

    print(f"Epoch {epoch+1} Average Loss: {total_loss / len(dataloader):.4f}")

print("Training Complete!")

Starting training for bearing...
Epoch 1 | Batch 0/250 | Loss: 19.0099
Epoch 1 | Batch 50/250 | Loss: 15.5148
Epoch 1 | Batch 100/250 | Loss: 8.7544
Epoch 1 | Batch 150/250 | Loss: 14.1986
Epoch 1 | Batch 200/250 | Loss: 8.1315
Epoch 1 Average Loss: 8.5431
Epoch 2 | Batch 0/250 | Loss: 0.0061
Epoch 2 | Batch 50/250 | Loss: 0.0218
Epoch 2 | Batch 100/250 | Loss: 5.2324
Epoch 2 | Batch 150/250 | Loss: 0.0006
Epoch 2 | Batch 200/250 | Loss: 0.1433
Epoch 2 Average Loss: 1.3168
Epoch 3 | Batch 0/250 | Loss: 0.0007
Epoch 3 | Batch 50/250 | Loss: 0.0011
Epoch 3 | Batch 100/250 | Loss: 0.0008
Epoch 3 | Batch 150/250 | Loss: 6.0265
Epoch 3 | Batch 200/250 | Loss: 2.4179
Epoch 3 Average Loss: 0.8839
Epoch 4 | Batch 0/250 | Loss: 0.0025
Epoch 4 | Batch 50/250 | Loss: 0.2278
Epoch 4 | Batch 100/250 | Loss: 0.0025
Epoch 4 | Batch 150/250 | Loss: 0.0007
Epoch 4 | Batch 200/250 | Loss: 0.0017
Epoch 4 Average Loss: 1.0836
Epoch 5 | Batch 0/250 | Loss: 0.0110
Epoch 5 | Batch 50/250 | Loss: 0.0055
Epoch

In [35]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

model.eval()
train_features = []

print("Building Normal Feature Library...")
with torch.no_grad():
    for wavs, _ in dataloader:
        mels = torch.stack([preprocess_audio(w) for w in wavs]).to(device)
        # Without labels, the model returns the 768-dim embedding
        feats = model(mels) 
        train_features.append(feats.cpu().numpy())

train_features = np.vstack(train_features)

# Wang uses k=2 for the k-Nearest Neighbors distance
knn = NearestNeighbors(n_neighbors=1, metric='cosine')
knn.fit(train_features)
print("Library Ready.")

Building Normal Feature Library...
Library Ready.


In [36]:
# 1. Load Test Data
test_dataset = DCASEWangDataset("data/dcase2023t2/dev_data/raw", "bearing", is_train=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

test_scores = []
filenames = []

print("Scoring test files...")
with torch.no_grad():
    for wavs, _ in test_loader:
        mels = torch.stack([preprocess_audio(w) for w in wavs]).to(device)
        feats = model(mels).cpu().numpy()
        
        # Calculate distance to nearest normal neighbor
        distances, _ = knn.kneighbors(feats)
        # Anomaly Score is the mean distance to the k-neighbors
        scores = np.mean(distances, axis=1)
        test_scores.extend(scores)

# Save results to CSV
results = pd.DataFrame({
    'file_name': [os.path.basename(f) for f in test_dataset.file_list],
    'anomaly_score': test_scores
})
results.to_csv(f"results_{dataset.machine_type}.csv", index=False)
print("Results saved to CSV!")

Scoring test files...
Results saved to CSV!


In [38]:
import pandas as pd
import numpy as np
from sklearn import metrics

# 1. Load your results
df = pd.read_csv('results_bearing.csv')

# 2. Extract Ground Truth from filenames
# In DCASE, 'anomaly' is the positive class (1) and 'normal' is (0)
df['label'] = df['file_name'].apply(lambda x: 1 if 'anomaly' in x else 0)

# 3. Calculate AUC
auc = metrics.roc_auc_score(df['label'], df['anomaly_score'])

# 4. Calculate pAUC (Standard DCASE setting: max_fpr=0.1)
p_auc = metrics.roc_auc_score(df['label'], df['anomaly_score'], max_fpr=0.1)

print(f"Machine Type: Bearing")
print(f"AUC: {auc:.4f}")
print(f"pAUC (fpr=0.1): {p_auc:.4f}")

Machine Type: Bearing
AUC: 0.6865
pAUC (fpr=0.1): 0.6137
